In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/car-price-turkiye-fiat-egea-2-el-fiyatlar/fiategea.csv


In [2]:
df = pd.read_csv('/kaggle/input/car-price-turkiye-fiat-egea-2-el-fiyatlar/fiategea.csv')
df.head(5)

,marka,model,motorhacmi,motortipi,paket,yil,km,renk,fiyat,sehir
0,Fiat,Egea,1.4,Fire,Urban,2021,101850.00,Beyaz,858000.0,İzmir
1,Fiat,Egea,1.3,Multijet,Easy,2021,195700.00,Beyaz,679000.0,Gaziantep
2,Fiat,Egea,1.6,Multijet,Urban,2025,5820.97,Beyaz,1188250.0,Mersin
3,Fiat,Egea,1.3,Multijet,Easy,2023,17640.00,Beyaz,872030.0,İstanbul
4,Fiat,Egea,1.3,Multijet,Urban,2021,74880.00,Kırmızı,892400.0,Rize


In [3]:
df = df.drop(columns = ['marka' , 'model' , 'sehir'])
df.head(5)

,motorhacmi,motortipi,paket,yil,km,renk,fiyat
0,1.4,Fire,Urban,2021,101850.00,Beyaz,858000.0
1,1.3,Multijet,Easy,2021,195700.00,Beyaz,679000.0
2,1.6,Multijet,Urban,2025,5820.97,Beyaz,1188250.0
3,1.3,Multijet,Easy,2023,17640.00,Beyaz,872030.0
4,1.3,Multijet,Urban,2021,74880.00,Kırmızı,892400.0


In [4]:
df = pd.get_dummies(df , columns=['motortipi' , 'paket' , 'renk'], drop_first = True , dtype = int)
df.head(5)

,motorhacmi,yil,km,fiyat,motortipi_Fire,motortipi_Multijet,motortipi_T4,paket_Easy,paket_Lounge,paket_Mirror,...,renk_Kahverengi,renk_Kırmızı,renk_Lacivert,renk_Mavi,renk_Mavi (metalik),renk_Siyah,renk_Turkuaz,renk_Yeşil,renk_Yeşil (metalik),renk_Şampanya
0,1.4,2021,101850.00,858000.0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1.3,2021,195700.00,679000.0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1.6,2025,5820.97,1188250.0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1.3,2023,17640.00,872030.0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1.3,2021,74880.00,892400.0,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [5]:
df['arac_yasi'] = 2026 - df['yil']
df = df.drop(columns = ['yil'])
df.head(5)

,motorhacmi,km,fiyat,motortipi_Fire,motortipi_Multijet,motortipi_T4,paket_Easy,paket_Lounge,paket_Mirror,paket_Street,...,renk_Kırmızı,renk_Lacivert,renk_Mavi,renk_Mavi (metalik),renk_Siyah,renk_Turkuaz,renk_Yeşil,renk_Yeşil (metalik),renk_Şampanya,arac_yasi
0,1.4,101850.00,858000.0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5
1,1.3,195700.00,679000.0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,5
2,1.6,5820.97,1188250.0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,1.3,17640.00,872030.0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,3
4,1.3,74880.00,892400.0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,5


In [6]:
y = df['fiyat'].values
x = df.drop(columns = ['fiyat'] , axis = 1).values
from sklearn.model_selection import train_test_split
x_train , x_test , y_train , y_test = train_test_split(x, y , test_size=0.33 , random_state=42)


In [7]:
from sklearn.preprocessing import StandardScaler
x_sc = StandardScaler()

x_train = x_sc.fit_transform(x_train)
x_test = x_sc.transform(x_test)

y_sc = StandardScaler()

y_train = y_sc.fit_transform(y_train.reshape(-1,1))
y_test  = y_sc.transform(y_test.reshape(-1,1))

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential()

# Katmanları biraz daha genişletelim ki Egea'nın paket/renk/yaş kombinasyonlarını çözebilsin
model.add(Dense(units=64, activation='relu', input_shape=(x_train.shape[1],)))
model.add(Dense(units=32, activation='relu'))
model.add(Dropout(0.2))

# Çıkış katmanı
model.add(Dense(units=1 , activation = 'linear'))
# Learning rate'i doğrudan burada veriyoruz
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model.summary()


# Eğitimi başlatıyoruz
history = model.fit(
    x_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)



2026-01-31 21:52:42.412856: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769896362.631410      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769896362.697012      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769896363.219827      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769896363.219889      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769896363.219892      17 computation_placer.cc:177] computation placer alr

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,969 (15.50 KB)

 Trainable params: 3,969 (15.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 1.0112 - mae: 0.7780 - val_loss: 0.5343 - val_mae: 0.5665
Epoch 2/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6708 - mae: 0.6057 - val_loss: 0.3595 - val_mae: 0.4572
Epoch 3/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.4045 - mae: 0.4505 - val_loss: 0.2877 - val_mae: 0.4189
Epoch 4/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3983 - mae: 0.4639 - val_loss: 0.2451 - val_mae: 0.3857
Epoch 5/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3834 - mae: 0.4276 - val_loss: 0.2297 - val_mae: 0.3715
Epoch 6/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3620 - mae: 0.4148 - val_loss: 0.2197 - val_mae: 0.3611
Epoch 7/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2616 - mae: 0.3750 - val_loss: 0.2148 - val_mae: 0.3533
Epoch 8/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2644 - mae: 0.3692 - val_loss: 0.2084 - val_mae: 0.3448
Epoch 9/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.27

In [9]:
# -------------------------------------------------
# 1️⃣2️⃣ TEST DEĞERLENDİRMESİ (SCALE'LI)
# -------------------------------------------------
test_loss, test_mae = model.evaluate(x_test, y_test)
print("Scaled Test MAE:", test_mae)

# -------------------------------------------------
# 1️⃣3️⃣ GERÇEK TL MAE HESABI (🔥 EN ÖNEMLİ 🔥)
# -------------------------------------------------
# GPT'nin hatasını düzelten gerçek TL MAE hesabı:
y_pred_scaled = model.predict(x_test)
y_pred = y_sc.inverse_transform(y_pred_scaled) # Tahminler TL oldu
y_test_real = y_sc.inverse_transform(y_test)   # Gerçek fiyatlar da TL oldu (EKSİK BURASIYDI)

real_mae = np.mean(np.abs(y_pred.flatten() - y_test_real.flatten()))
print("Gerçek MAE (TL):", real_mae)


11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2468 - mae: 0.3625 
Scaled Test MAE: 0.358838826417923
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
Gerçek MAE (TL): 55807.777


In [10]:
df.head(5)

,motorhacmi,km,fiyat,motortipi_Fire,motortipi_Multijet,motortipi_T4,paket_Easy,paket_Lounge,paket_Mirror,paket_Street,...,renk_Kırmızı,renk_Lacivert,renk_Mavi,renk_Mavi (metalik),renk_Siyah,renk_Turkuaz,renk_Yeşil,renk_Yeşil (metalik),renk_Şampanya,arac_yasi
0,1.4,101850.00,858000.0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5
1,1.3,195700.00,679000.0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,5
2,1.6,5820.97,1188250.0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,1.3,17640.00,872030.0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,3
4,1.3,74880.00,892400.0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,5
